# Feature Tokenizer Transformer

- FT-Transformer with continuous and discretised feature tokenisation
- Benchmark models: XGBoost, LightGBM, Fama-French five-factor Ridge classifier, MLP
- Rolling expanding-window evaluation with per-window no-lookahead preprocessing
- Interpretability: Attention Rollout, Integrated Gradients, GradientSHAP, TreeSHAP
- Attribution temporal stability analysis and Fama-French factor alignment

### packages

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math
from typing import Optional

## Transformer

In [ ]:
class ScalarTokeniser(nn.Module):
    # Feature k gets its own weight vector W_k of size d_model
    # Token_k = x_k * W_k + b_k
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, F]  ->  [B, F, D]
        return x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)

#TODO: Add Categorical Tokenizer

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_h = d_model // n_heads
        self.scale = self.d_h ** -0.5

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.drop = nn.Dropout(dropout)

        # Attention weights are cached after every forward pass.
        # The rollout method reads this buffer without re-running the model.
        self._cached_weights: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D = x.shape
        qkv = self.qkv(x).view(B, S, 3, self.n_heads, self.d_h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        weights = scores.softmax(dim=-1)
        self._cached_weights = weights.detach().clone()

        out = (self.drop(weights) @ v).transpose(1, 2).contiguous().view(B, S, D)
        return self.proj(out)

    @property
    def attention_weights(self) -> Optional[torch.Tensor]:
        return self._cached_weights


class ReGLU(nn.Module):
    # Rectified Gated Linear Unit. Splits the channel dimension in half
    # and uses the second half as a ReLU gate for the first half.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff * 2)   # x2 because ReGLU halves channels
        self.act = ReGLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.drop(self.act(self.fc1(x))))


class EncoderBlock(nn.Module):
    # Pre-LN layout: normalise before attention and FFN, not after.
    # Empirically more stable than post-LN for small tabular Transformers
    # where token sequence length is short.
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x